### Evaluating the finetuned LLM

In [1]:
import psutil

def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running

ollama_running = check_if_running("ollama")

if not ollama_running:
    raise RuntimeError("Ollama not running. Launch ollama before proceeding.")
print("Ollama running:", check_if_running("ollama"))

Ollama running: True


In [2]:
import json
from tqdm import tqdm

file_path = "instruction-data-with-response.json"

with open(file_path, "r") as file:
    test_data = json.load(file)


def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [4]:
import requests  # noqa: F811
# import urllib.request

def query_model(
    prompt,
    model="llama3.2:latest",
    # If you used OLLAMA_HOST=127.0.0.1:11435 ollama serve
    # update the address from 11434 to 11435
    url="http://localhost:11434/api/chat"
):
    # Create the data payload as a dictionary
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "options": {     # Settings below are required for deterministic responses
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }

    with requests.post(url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()
        response_data = ""
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            response_json = json.loads(line)
            if "message" in response_json:
                response_data += response_json["message"]["content"]

    return response_data


model = "llama3.2:latest"
result = query_model("What do Llamas eat?", model)
print(result)

Llamas are herbivores, which means they primarily eat plants and plant-based foods. Their diet typically consists of:

1. Grasses: Llamas love to graze on various types of grasses, including tall grasses, short grasses, and grasses with seeds.
2. Hay: Llamas are often fed hay, such as timothy hay, alfalfa hay, or other types of hay, as a supplement to their diet.
3. Grains: Llamas may also be fed grains like oats, barley, or corn, but these should be given in moderation.
4. Fruits and vegetables: Llamas enjoy fruits like apples, carrots, and sweet potatoes, as well as leafy greens like kale and spinach.
5. Browse: Llamas may also eat browse, which includes twigs, branches, and leaves from trees and shrubs.

It's essential to note that llamas have a unique digestive system, which allows them to break down and extract nutrients from plant material that other animals might not be able to digest. However, this also means that they can be prone to digestive issues if their diet is not balan

In [5]:
for entry in test_data[:3]:
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"score the model response `{entry['model_response']}`"
        f" on a scale from 0 to 100, where 100 is the best score. "
    )
    print("\nDataset response:")
    print(">>", entry['output'])
    print("\nModel response:")
    print(">>", entry["model_response"])
    print("\nScore:")
    print(">>", query_model(prompt))
    print("\n-------------------------")


Dataset response:
>> The car is as fast as lightning.

Model response:
>> The car is as fast as a bullet.

Score:
>> To rewrite the sentence using a simile, we need to compare the car's speed to something else. A bullet is a fast-moving object, so it's a good comparison.

Score: 80

The model response "The car is as fast as a bullet." is close to the correct answer "The car is as fast as lightning." because it uses a simile to compare the car's speed to a fast-moving object. However, lightning is generally considered to be even faster than a bullet, so the model response is not the best possible answer.

-------------------------

Dataset response:
>> The type of cloud typically associated with thunderstorms is cumulonimbus.

Model response:
>> The type of cloud associated with thunderstorms is a cumulus cloud.

Score:
>> Score: 0

The model response is incorrect because it incorrectly identifies cumulus clouds as the type associated with thunderstorms. Cumulus clouds are actually ass

In [7]:
def generate_model_scores(json_data, json_key, model="llama3.2:latest"):
    scores = []
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."
        )
        score = query_model(prompt, model)
        try:
            scores.append(int(score))
        except ValueError:
            print(f"Could not convert score: {score}")
            continue

    return scores


scores = generate_model_scores(test_data, "model_response")
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores)/len(scores):.2f}\n")

Scoring entries:   9%|▉         | 10/110 [00:27<04:01,  2.41s/it]

Could not convert score: 50

The model response is partially correct, but it incorrectly includes 14 in the list of prime numbers. The correct classification is:

Prime numbers: 11, 19
Composite numbers: 14


Scoring entries:  15%|█▍        | 16/110 [00:41<03:56,  2.51s/it]

Could not convert score: 90

Explanation: The model response is mostly correct, but it's missing the classification for "wood". The correct response should be "Solid: Mercury
Liquid: Oxygen
Gas: Wood". The model response is missing the classification for "wood" which is a solid. Therefore, the score is 90, indicating that the model response is mostly correct but needs improvement in handling the classification for "wood".


Scoring entries:  16%|█▋        | 18/110 [00:45<03:36,  2.35s/it]

...

Could not convert score: The classification of the sentence 'Please open the door.' is imperative.

Score: 100


Scoring entries:  99%|█████████▉| 109/110 [04:11<00:02,  2.29s/it]

Could not convert score: 60

She never forgets to call.


Scoring entries: 100%|██████████| 110/110 [04:13<00:00,  2.30s/it]

Number of scores: 98 of 110
Average score: 47.40

